# Lab | Music recommendations

- First re-run everything in this notebook to ensure you're comfortable with the concepts of similar audio recommendation systems based on RAG.
- Using music datasets from [this](https://github.com/Yuan-ManX/ai-audio-datasets?tab=readme-ov-file#m) github repo, create a local RAG to recommend sons based on users preferences. Example dataset from that link could be [this](https://zenodo.org/records/5794629) Artificial multitrack audio data. Feel free to find you're own datasets online, or combine the dataset used in this lab with a few you found to make some recommendations.
- Go ahead and build something great in 4 hours.

This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [3]:
# Part 1 (ESC-50 demo) + Part 2 (music RAG) dependencies.
# datasets<4 keeps the classic `audio["array"]` decoding (v4 switched to torchcodec).
!pip install -qU "datasets<4" pinecone panns-inference librosa soundfile openai python-dotenv pandas tqdm

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.35 requires openai<3.0.0,>=1.104.2, but you have openai 3.42.1 which is incompatible.

[notice] A new release of pip is available: 25.1.1 -> 26.2.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [1]:
# panns_inference downloads its label file + checkpoint with `wget` on first import.
# macOS/Windows often don't have wget, so fetch them with Python instead (no-op if already there).
from pathlib import Path
import urllib.request

PANNS_DIR = Path.home() / "panns_data"
PANNS_DIR.mkdir(exist_ok=True)
PANNS_FILES = {
    "class_labels_indices.csv": "https://raw.githubusercontent.com/qiuqiangkong/audioset_tagging_cnn/master/metadata/class_labels_indices.csv",
    "Cnn14_mAP=0.431.pth": "https://zenodo.org/record/3987831/files/Cnn14_mAP%3D0.431.pth?download=1",  # ~330 MB
}
for name, url in PANNS_FILES.items():
    dest = PANNS_DIR / name
    if not dest.exists() or dest.stat().st_size == 0:
        print("Downloading", name, "...")
        urllib.request.urlretrieve(url, dest)
print("PANNs files ready in", PANNS_DIR)

PANNs files ready in C:\Users\Roberto\panns_data


# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [4]:
from datasets import load_dataset

# load the dataset from huggingface model hub
data = load_dataset("ashraq/esc50", split="train")
data

README.md:   0%|          | 0.00/345 [00:00<?, ?B/s]

C:\Users\Roberto\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Roberto\.cache\huggingface\hub\datasets--ashraq--esc50. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

C:\Users\Roberto\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Roberto\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00002-2f1ab7b824ec75(…): reconstructing file:   0%|          |  0.00B /  387MB            

data/train-00000-of-00002-2f1ab7b824ec75(…): downloading bytes:           |  0.00B            

data/train-00001-of-00002-27425e5c1846b4(…): reconstructing file:   0%|          |  0.00B /  387MB            

data/train-00001-of-00002-27425e5c1846b4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
    num_rows: 2000
})

The audios in the dataset are sampled at 44100Hz and loaded into NumPy arrays. Let's take a look.

In [5]:
# select the audio feature and display top three
audios = data["audio"]
audios[:3]

[{'path': None,
  'array': array([0., 0., 0., ..., 0., 0., 0.], shape=(220500,)),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.01184082, -0.10336304, -0.14141846, ...,  0.06985474,
          0.04049683,  0.00274658], shape=(220500,)),
  'sampling_rate': 44100},
 {'path': None,
  'array': array([-0.00695801, -0.01251221, -0.01126099, ...,  0.215271  ,
         -0.00875854, -0.28903198], shape=(220500,)),
  'sampling_rate': 44100}]

We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [6]:
import numpy as np

# select only the audio data from the dataset and store in a numpy array
# (float32 halves the memory vs. the default float64: 2000 x 5 s x 44.1 kHz)
audios = np.array([a["array"] for a in data["audio"]], dtype=np.float32)

# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [7]:
import torch
from panns_inference import AudioTagging

# panns_inference only knows 'cuda' or 'cpu' -- anything else (e.g. 'mps') silently falls back to CPU.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = AudioTagging(checkpoint_path=None, device=DEVICE)

Checkpoint path: C:\Users\Roberto/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [14]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
# initialize connection to pinecone (get API key at app.pinecone.io)
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY") or "YOUR_API_KEY"

In [15]:
import os
from pinecone import Pinecone

# configure client
pc = Pinecone(api_key=PINECONE_API_KEY)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [16]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

Create the index:

In [17]:
index_name = "audio-search-demo"

In [18]:
import time

# check if index already exists (it shouldn't if this is first time)
if index_name not in pc.list_indexes().names():
    # if does not exist, create index
    pc.create_index(
        index_name,
        dimension=2048,
        metric='cosine',
        spec=spec
    )
    # wait for index to be initialized
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1)

# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=2048, total_vector_count=0, metric='cosine', namespaces=0)

# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [19]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(audios), batch_size)):
    # find end of batch
    i_end = min(i+batch_size, len(audios))
    # extract batch
    batch = audios[i:i_end]
    # generate embeddings for all the audios in the batch
    _, emb = model.inference(batch)
    # create unique IDs
    ids = [f"{idx}" for idx in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb.tolist()))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()

  0%|          | 0/32 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=2048, total_vector_count=2000, metric='cosine', namespaces=1)

We now have *2000* audio records indexed in Pinecone, we're ready to begin querying.

# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [20]:
from IPython.display import Audio, display

# we set an audio number to select from the dataset
audio_num = 400
# get the audio data of the audio number
query_audio = data[audio_num]["audio"]["array"]
# get the category of the audio number
category = data[audio_num]["category"]
# print the category and play the audio
print("Query Audio:", category)
Audio(query_audio, rate=44100)

Query Audio: car_horn


<audio player removed to keep the notebook small; re-run the cell to listen>

We have got the sound of a car horn. Let's generate an embedding for this sound.

In [21]:
# reshape query audio
query_audio = query_audio[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)
xq.shape

(1, 2048)

We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [22]:
# query pinecone index with the query audio embeddings
# (current Pinecone clients want a flat list, so take the single row of the (1, 2048) array)
results = index.query(vector=xq[0].tolist(), top_k=3)
results

QueryResponse(matches=[ScoredVector(id='400', score=1.00000453, values=[]), ScoredVector(id='1667', score=0.842127204, values=[]), ScoredVector(id='1666', score=0.831753731, values=[])], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Mon, 02 Nov 2026 16:02:12 GMT', 'content-type': 'application/json', 'content-length': '218', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '32', 'x-pinecone-request-latency-ms': '103', 'x-envoy-upstream-service-time': '96', 'x-pinecone-response-duration-ms': '105', 'grpc-status': '0', 'server': 'envoy'}))

Notice that the top result is the audio number 400 from our dataset, which is our query audio (the most similar item should always be the query itself). Let's listen to the top three results. 

In [23]:
# play the top 3 similar audios
for r in results["matches"]:
    # select the audio data from the databse using the id as an index
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

We have great results, everything aligns with what seems to be a busy city street with car horns.

Let's write a helper function to run the queries using audio from our dataset easily. We do not need to embed these audio samples again as we have already, they are just stored in Pinecone. So, we specify the `id` of the query audio to search with and tell Pinecone to search with that.

In [24]:
def find_similar_audios(id):
    print("Query Audio:")
    # select the audio data from the databse using the id as an index
    query_audio = data[id]["audio"]["array"]
    # play the query audio
    display(Audio(query_audio, rate=44100))
    # query pinecone index with the query audio id
    result = index.query(id=str(id), top_k=5)
    print("Result:")
    # play the top 5 similar audios
    for r in result["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=44100))

In [25]:
find_similar_audios(1642)

Query Audio:


<audio player removed to keep the notebook small; re-run the cell to listen>

Result:


<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

Here we return a set of revving motors (they seem to either be vehicles or lawnmowers).

In [26]:
find_similar_audios(452)

Query Audio:


<audio player removed to keep the notebook small; re-run the cell to listen>

Result:


<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

And now a more relaxing set of birds chirping in nature.

Let's use another audio sample from elsewhere (eg not this dataset) and see how the search performs with this.

In [27]:
import os
os.makedirs("data", exist_ok=True)
if not os.path.exists("data/miaow_16k.wav"):
    urllib.request.urlretrieve("https://storage.googleapis.com/audioset/miaow_16k.wav", "data/miaow_16k.wav")

We can load the audio into a Numpy array as follows:

In [28]:
import librosa

a, _ = librosa.load("./data/miaow_16k.wav", sr=44100)
Audio(a, rate=44100)

<audio player removed to keep the notebook small; re-run the cell to listen>

Now we generate the embeddings for this audio and query the Pinecone index.

In [29]:
# reshape query audio
query_audio = a[None, :]
# get the embeddings for the audio from the model
_, xq = model.inference(query_audio)

# query pinecone index with the query audio embeddings
results = index.query(vector=xq[0].tolist(), top_k=3)

# play the top 3 similar audios
for r in results["matches"]:
    a = data[int(r["id"])]["audio"]["array"]
    display(Audio(a, rate=44100))

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

<audio player removed to keep the notebook small; re-run the cell to listen>

Our audio search application has identified a set of similar cat sounds, which is excellent.

# Delete the Index

Delete the index once you are sure that you do not want to use it anymore. Once the index is deleted, you cannot use it again.

In [30]:
pc.delete_index(index_name)

---
# Part 2 — Local music RAG recommender (with GPT Luna)

Now the actual lab: recommend **songs** from a user's preferences, fully locally for retrieval, with an LLM for the "augmented generation" step.

**Dataset:** [`lewtun/music_genres_small`](https://huggingface.co/datasets/lewtun/music_genres_small) — 1,000 music clips with a `genre` label (parquet, loads straight from the Hub). Swap in any dataset with an `audio` column and a label column.

**How it works**

| Step | What happens | Tooling |
|---|---|---|
| Index | Each clip → PANNs **2048‑d embedding** (sound similarity) + **527 AudioSet tag probabilities** (e.g. *Piano*, *Electric guitar*, *Male singing*, *Jazz*, *Sad music*) | `panns_inference`, NumPy |
| Store | Normalised vectors + metadata in a small **local vector index** (cached to `data/music_index.npz`) | NumPy |
| Understand | User's free‑text request → structured preferences (genres, wanted tags, unwanted tags), constrained to the real tag vocabulary | **GPT Luna** (structured output) |
| Retrieve | Hybrid score = tag match + genre match + audio similarity to songs the user liked | NumPy |
| Generate | LLM picks a diverse top‑N **only from retrieved candidates** and explains each pick from its tags | **GPT Luna** |

No `OPENAI_API_KEY`? Everything still runs: steps 3 and 5 fall back to keyword matching and template explanations.

> Part 2 is independent of Pinecone. If memory is tight, restart the kernel and run only the install, PANNs‑download, model and `.env` cells before this section.

In [31]:
import os, json, re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import Audio, display
from datasets import load_dataset, Audio as HFAudio
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# ---- LLM config -------------------------------------------------------------
# GPT Luna = OpenAI's fast/cheap tier. "gpt-6-luna" is current; "gpt-5.6-luna" also works.
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
# Luna supports none | low | medium (default) | high | xhigh | max.
# "low" is plenty for parsing a request + ranking 15 candidates, and keeps it fast/cheap.
REASONING_EFFORT = os.getenv("OPENAI_REASONING_EFFORT", "low")

# ---- Audio config -------------------------------------------------------------
MUSIC_SR = 32000        # PANNs Cnn14 was trained on 32 kHz audio
CLIP_SECONDS = 10       # embed a centred 10 s window of each track
CACHE_PATH = "data/music_index.npz"
os.makedirs("data", exist_ok=True)

# Load the embedding model if Part 1 wasn't run in this kernel
if "model" not in globals():
    import torch
    from panns_inference import AudioTagging
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    model = AudioTagging(checkpoint_path=None, device=DEVICE)

## Load the music dataset
`cast_column(..., sampling_rate=32000)` makes 🤗 Datasets resample every clip on the fly to what PANNs expects (Part 1 fed 44.1 kHz audio to a 32 kHz model — fine for a demo, but slightly off).

In [32]:
music = load_dataset("lewtun/music_genres_small", split="train")
music = music.cast_column("audio", HFAudio(sampling_rate=MUSIC_SR))

GENRES = sorted(set(music["genre"]))
print(music)
print("Genres:", GENRES)
pd.Series(music["genre"]).value_counts()

README.md:   0%|          | 0.00/487 [00:00<?, ?B/s]

C:\Users\Roberto\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Roberto\.cache\huggingface\hub\datasets--lewtun--music_genres_small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001-63d68663287b16(…): reconstructing file:   0%|          |  0.00B /  391MB            

data/train-00000-of-00001-63d68663287b16(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['audio', 'song_id', 'genre_id', 'genre'],
    num_rows: 1000
})
Genres: ['Chiptune / Glitch', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'International', 'Pop', 'Punk', 'Rock']


International        100
Pop                  100
Electronic           100
Chiptune / Glitch    100
Punk                 100
Folk                 100
Hip-Hop              100
Rock                 100
Instrumental         100
Experimental         100
Name: count, dtype: int64

In [33]:
def to_clip(y, seconds=CLIP_SECONDS, sr=MUSIC_SR):
    """Mono float32, centre-cropped (or zero-padded) to a fixed length."""
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=0)
    n = int(seconds * sr)
    if len(y) >= n:
        start = (len(y) - n) // 2
        return y[start:start + n]
    return np.pad(y, (0, n - len(y)))

## Embed every track (once) and cache it
For each clip PANNs returns two things we use:
- `embedding` (2048‑d) → "sounds like" similarity
- `clipwise_output` (527 AudioSet class probabilities) → human‑readable **tags** the LLM can reason about

This is the slow step (a few minutes on CPU); the result is cached so re-runs are instant.

In [34]:
def embed_dataset(ds, batch_size=16):
    tag_list, emb_list = [], []
    for i in tqdm(range(0, len(ds), batch_size)):
        batch = ds[i:i + batch_size]["audio"]
        x = np.stack([to_clip(a["array"]) for a in batch])
        tags, emb = model.inference(x)
        tag_list.append(tags)
        emb_list.append(emb)
    return np.concatenate(emb_list), np.concatenate(tag_list)

if os.path.exists(CACHE_PATH) and np.load(CACHE_PATH)["emb"].shape[0] == len(music):
    cache = np.load(CACHE_PATH)
    music_emb, music_tagprobs = cache["emb"], cache["tags"]
    print("Loaded cached embeddings:", music_emb.shape)
else:
    music_emb, music_tagprobs = embed_dataset(music)
    np.savez(CACHE_PATH, emb=music_emb, tags=music_tagprobs)
    print("Embedded and cached:", music_emb.shape, music_tagprobs.shape)

  0%|          | 0/63 [00:00<?, ?it/s]

Embedded and cached: (1000, 2048) (1000, 527)


## Build the catalogue (metadata) and a local vector index
We keep only the **music‑related** AudioSet classes (singing types, instruments, genres, moods — ~155 of the 527) as the tag vocabulary.
Tag scores are z‑normalised per tag, because raw probabilities live on very different scales (almost everything scores high on *Pop music*, almost nothing on *Theremin*).

In [35]:
from panns_inference import labels as AUDIOSET_LABELS

_ix = AUDIOSET_LABELS.index
MUSIC_TAG_IDX = np.r_[_ix("Singing"):_ix("Humming") + 1,                         # vocals: singing, choir, rapping...
                      _ix("Plucked string instrument"):_ix("Scary music") + 1]   # instruments, genres, moods
MUSIC_TAGS = [AUDIOSET_LABELS[i] for i in MUSIC_TAG_IDX]
TAG_POS = {t: j for j, t in enumerate(MUSIC_TAGS)}

tag_probs = music_tagprobs[:, MUSIC_TAG_IDX]
tag_z = (tag_probs - tag_probs.mean(0)) / (tag_probs.std(0) + 1e-6)
tag_z = np.clip(tag_z, -3, 3)   # so one outlier tag can't drown everything else

def describe(i, k=5):
    """Top-k tags of a track: most distinctive (z-score) rather than most common."""
    return [MUSIC_TAGS[j] for j in np.argsort(-tag_z[i])[:k]]

catalog = pd.DataFrame({
    "song_id": music["song_id"],
    "genre": music["genre"],
})
catalog["tags"] = [describe(i) for i in range(len(catalog))]
print(len(MUSIC_TAGS), "music tags, e.g.", MUSIC_TAGS[:8])
catalog.head()

155 music tags, e.g. ['Singing', 'Choir', 'Yodeling', 'Chant', 'Mantra', 'Male singing', 'Female singing', 'Child singing']


,song_id,genre,tags
0,11483,International,"[Yodeling, Change ringing (campanology), Opera..."
1,6824,International,"[Synthesizer, Sampler, Electronic music, Dubst..."
2,8361,Pop,"[Soul music, Rhythm and blues, Song, Tender mu..."
3,12269,International,"[Drum and bass, Electronica, House music, Samp..."
4,1063,Electronic,"[Afrobeat, Background music, Happy music, Elec..."


In [36]:
class LocalVectorIndex:
    """A tiny in-memory cosine-similarity index (same idea as Pinecone, no server needed)."""

    def __init__(self, vectors, metadata: pd.DataFrame):
        v = np.asarray(vectors, dtype=np.float32)
        self.vectors = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-12)
        self.metadata = metadata.reset_index(drop=True)

    def similarity(self, vector):
        q = np.asarray(vector, dtype=np.float32).reshape(-1)
        return self.vectors @ (q / (np.linalg.norm(q) + 1e-12))

    def query(self, vector=None, id=None, top_k=5, exclude=(), genres=None):
        if id is not None:
            vector = self.vectors[id]
            exclude = set(exclude) | {id}
        s = self.similarity(vector)
        if genres:
            s = np.where(self.metadata["genre"].isin(genres).to_numpy(), s, -np.inf)
        s[list(exclude)] = -np.inf
        top = [i for i in np.argsort(-s)[:top_k] if np.isfinite(s[i])]
        return self.metadata.iloc[top].assign(score=s[top].round(3))

music_index = LocalVectorIndex(music_emb, catalog)

## Sanity check: does "sounds similar" track genre?
For every track, look at its 5 nearest neighbours and count how many share its genre. Random guessing would score ≈ 1 / #genres.

In [37]:
sims = music_index.vectors @ music_index.vectors.T
np.fill_diagonal(sims, -np.inf)
nn = np.argsort(-sims, axis=1)[:, :5]
genre_arr = catalog["genre"].to_numpy()
p_at_5 = (genre_arr[nn] == genre_arr[:, None]).mean()
baseline = (catalog["genre"].value_counts(normalize=True) ** 2).sum()
print(f"Genre precision@5 of audio neighbours: {p_at_5:.2f}   (random baseline ≈ {baseline:.2f})")
del sims

Genre precision@5 of audio neighbours: 0.43   (random baseline ≈ 0.10)


In [38]:
def play(ids, scores=None):
    for k, i in enumerate(ids):
        row = catalog.iloc[int(i)]
        extra = f" | score {scores[k]:.2f}" if scores is not None else ""
        print(f"#{int(i)} | {row.genre} | {', '.join(row.tags[:4])}{extra}")
        display(Audio(to_clip(music[int(i)]["audio"]["array"]), rate=MUSIC_SR))

# Plain audio-to-audio search, like Part 1 but on music
seed = 0
print("Seed track:"); play([seed])
print("\nSounds most similar:")
res = music_index.query(id=seed, top_k=3)
play(res.index, res.score.to_numpy())

Seed track:
#0 | International | Yodeling, Change ringing (campanology), Opera, Bicycle bell


<audio player removed to keep the notebook small; re-run the cell to listen>


Sounds most similar:
#950 | International | Male singing, Chant, Child singing, A capella | score 0.88


<audio player removed to keep the notebook small; re-run the cell to listen>

#496 | International | Tuning fork, Bicycle bell, Tabla, Flute | score 0.78


<audio player removed to keep the notebook small; re-run the cell to listen>

#76 | Experimental | Shofar, Tuning fork, Wind instrument, woodwind instrument, Yodeling | score 0.76


<audio player removed to keep the notebook small; re-run the cell to listen>

## The LLM layer (GPT Luna)
One helper calls the **Responses API** with a strict JSON schema, so the model can only answer with tags/genres that actually exist in our index.

Notes for GPT Luna:
- uses `reasoning={"effort": ...}` — `temperature` is only accepted when effort is `"none"`, so we don't send it;
- change `OPENAI_MODEL` (env or the config cell) to try `gpt-5.6-luna`, `gpt-6-sol`, etc. — nothing else changes.

In [39]:
llm = None
if OPENAI_API_KEY:
    from openai import OpenAI
    llm = OpenAI(api_key=OPENAI_API_KEY)
    print(f"LLM enabled: {LLM_MODEL} (reasoning effort: {REASONING_EFFORT})")
else:
    print("No OPENAI_API_KEY found -> using the keyword fallback (no LLM).")

def ask_llm_json(instructions: str, user_input: str, schema: dict, name: str) -> dict:
    resp = llm.responses.create(
        model=LLM_MODEL,
        reasoning={"effort": REASONING_EFFORT},
        instructions=instructions,
        input=user_input,
        text={"format": {"type": "json_schema", "name": name, "schema": schema, "strict": True}},
    )
    return json.loads(resp.output_text)

LLM enabled: gpt-6-luna (reasoning effort: low)


### Step 1 — understand the request
Free text → `{genres, exclude_genres, include_tags, exclude_tags}`, each constrained to our vocabulary via `enum`.

In [40]:
def _str_list(values):
    return {"type": "array", "items": {"type": "string", "enum": values}}

PREF_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "genres": _str_list(GENRES),
        "exclude_genres": _str_list(GENRES),
        "include_tags": _str_list(MUSIC_TAGS),
        "exclude_tags": _str_list(MUSIC_TAGS),
    },
    "required": ["genres", "exclude_genres", "include_tags", "exclude_tags"],
}

PREF_INSTRUCTIONS = f"""You translate a listener's music request into search filters for a song catalogue.
Catalogue genres: {GENRES}.
Each song is auto-tagged with AudioSet music tags (instruments, vocal types, styles, moods).
- genres / exclude_genres: only genres the user clearly wants or rejects (empty if unspecified).
- include_tags: 1-6 tags that best capture what the user wants to hear. Map vibes to concrete tags
  (e.g. "chill" -> "Ambient music", "Soft music"... ; "party" -> "Dance music", "Drum"...).
- exclude_tags: tags the user explicitly does not want (e.g. "no vocals" -> "Singing", "Male singing", "Female singing")."""

def parse_preferences_fallback(text: str) -> dict:
    """Keyword matcher used when no LLM is available."""
    t = text.lower()
    neg = re.findall(r"\b(?:no|without|not|avoid|hate)\s+((?:[a-z\-]+\s?){1,3})", t)  # up to 3 words after a negation
    neg_text = " ".join(neg)
    match = lambda name: re.search(rf"\b{re.escape(name.lower())}\b", t) is not None
    genres = [g for g in GENRES if match(g)]
    tags = [m for m in MUSIC_TAGS if match(m) or match(m.replace(" music", ""))]
    excl_tags = [m for m in tags if m.lower().replace(" music", "") in neg_text]
    if any(w in neg_text for w in ("vocal", "singing", "voice")):
        excl_tags += [m for m in ("Singing", "Male singing", "Female singing", "Rapping") if m in TAG_POS]
    excl_genres = [g for g in genres if g.lower() in neg_text]
    return {
        "genres": [g for g in genres if g not in excl_genres],
        "exclude_genres": excl_genres,
        "include_tags": [m for m in tags if m not in excl_tags],
        "exclude_tags": sorted(set(excl_tags)),
    }

def parse_preferences(text: str) -> dict:
    if not text:
        return {"genres": [], "exclude_genres": [], "include_tags": [], "exclude_tags": []}
    if llm is None:
        return parse_preferences_fallback(text)
    return ask_llm_json(PREF_INSTRUCTIONS, text, PREF_SCHEMA, "music_preferences")

parse_preferences("calm piano music without vocals, maybe some jazz")

{'genres': ['Instrumental'],
 'exclude_genres': [],
 'include_tags': ['Piano', 'Jazz', 'Tender music'],
 'exclude_tags': ['Singing',
  'Male singing',
  'Female singing',
  'Child singing',
  'Synthetic singing']}

### Step 2 — retrieve (hybrid scoring, all local)
`score = mean z(wanted tags) − mean z(unwanted tags) + genre bonus + audio similarity to liked songs`.
Liked songs can be catalogue ids **or** your own audio file. Duplicate clips of the same `song_id` are collapsed.

In [41]:
def embed_audio(source):
    """Path to an audio file, or a raw waveform already at MUSIC_SR -> 2048-d embedding."""
    if isinstance(source, (str, os.PathLike)):
        import librosa
        source, _ = librosa.load(source, sr=MUSIC_SR, mono=True)
    _, emb = model.inference(to_clip(source)[None, :])
    return emb[0]

def _z(x):
    return (x - x.mean()) / (x.std() + 1e-6)

def retrieve(prefs, liked_ids=(), liked_audio=None, top_k=15, genre_bonus=3.0, audio_weight=1.0):
    n = len(catalog)
    score = np.zeros(n)
    inc = [TAG_POS[t] for t in prefs.get("include_tags", []) if t in TAG_POS]
    exc = [TAG_POS[t] for t in prefs.get("exclude_tags", []) if t in TAG_POS]
    if inc:
        score += tag_z[:, inc].mean(1)
    if exc:
        score -= tag_z[:, exc].mean(1)
    if prefs.get("genres"):
        score += genre_bonus * catalog["genre"].isin(prefs["genres"]).to_numpy()

    seeds = [music_index.vectors[i] for i in liked_ids]
    if liked_audio is not None:
        e = embed_audio(liked_audio)
        seeds.append(e / (np.linalg.norm(e) + 1e-12))
    if seeds:
        score += audio_weight * np.clip(_z(music_index.similarity(np.mean(seeds, axis=0))), -3, 3)

    if prefs.get("exclude_genres"):
        score[catalog["genre"].isin(prefs["exclude_genres"]).to_numpy()] = -np.inf
    score[list(liked_ids)] = -np.inf

    cands = (catalog.assign(score=score)
             .loc[lambda d: np.isfinite(d.score)]
             .sort_values("score", ascending=False)
             .drop_duplicates("song_id")
             .head(top_k))
    return cands

### Step 3 — generate: pick & explain (grounded in the retrieved candidates)
The LLM only sees the candidate list (id, genre, tags, score) and must answer with ids from it — anything else is dropped, and we top up from the ranking if needed. The dataset has no titles/artists, so the prompt forbids inventing them.

In [42]:
REC_SCHEMA = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "summary": {"type": "string"},
        "recommendations": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {"id": {"type": "integer"}, "reason": {"type": "string"}},
                "required": ["id", "reason"],
            },
        },
    },
    "required": ["summary", "recommendations"],
}

REC_INSTRUCTIONS = """You are a music recommender. You receive a listener request and CANDIDATE tracks retrieved
from a catalogue (id, genre, auto-detected tags, retrieval score — higher = better match).
Pick the {n} best tracks for the listener, preferring a bit of variety over near-duplicates.
Only use ids from the candidate list. The catalogue has no titles or artists: never invent them.
For each pick give one short sentence explaining the fit, grounded in its genre/tags.
Write a one-sentence summary of the selection."""

def _candidate_context(cands):
    return "\n".join(
        json.dumps({"id": int(i), "genre": r.genre, "tags": r.tags, "score": round(float(r.score), 2)})
        for i, r in cands.iterrows()
    )

def generate(request, prefs, cands, liked_ids=(), n=5):
    valid = list(cands.index)
    if llm is not None:
        liked_desc = [{"genre": catalog.genre[i], "tags": catalog.tags[i]} for i in liked_ids]
        user_input = (
            f"Listener request: {request or '(none — recommend from liked songs)'}\n"
            f"Liked songs: {json.dumps(liked_desc)}\n"
            f"Parsed preferences: {json.dumps(prefs)}\n\n"
            f"CANDIDATES:\n{_candidate_context(cands)}"
        )
        out = ask_llm_json(REC_INSTRUCTIONS.format(n=n), user_input, REC_SCHEMA, "recommendations")
        picks, seen = [], set()
        for r in out["recommendations"]:
            if r["id"] in valid and r["id"] not in seen:
                picks.append((r["id"], r["reason"])); seen.add(r["id"])
        summary = out["summary"]
    else:
        picks, summary = [], "Top matches by tag, genre and audio similarity."
    # top up (or fallback) straight from the retrieval ranking
    for i in valid:
        if len(picks) >= n:
            break
        if i not in {p for p, _ in picks}:
            r = cands.loc[i]
            picks.append((i, f"{r.genre} track with {', '.join(r.tags[:3])}."))
    return picks[:n], summary

### Put it together

In [43]:
def recommend(request: str = "", liked_ids=(), liked_audio=None, n=5, play_audio=True, show_prefs=True):
    liked_ids = [int(i) for i in liked_ids]
    prefs = parse_preferences(request)
    if show_prefs:
        print("Parsed preferences:", json.dumps(prefs))
    cands = retrieve(prefs, liked_ids, liked_audio, top_k=max(15, 3 * n))
    picks, summary = generate(request, prefs, cands, liked_ids, n)
    print("\n" + summary + "\n")
    out = catalog.loc[[i for i, _ in picks], ["song_id", "genre", "tags"]].assign(
        score=cands.loc[[i for i, _ in picks], "score"].round(2).to_numpy(),
        reason=[r for _, r in picks],
    )
    if play_audio:
        for (i, reason) in picks:
            print(f"#{i} | {catalog.genre[i]} | {reason}")
            display(Audio(to_clip(music[int(i)]["audio"]["array"]), rate=MUSIC_SR))
    return out

## Try it
**1. Text only** — describe what you're in the mood for.

In [44]:
recommend("Something calm and acoustic for studying - piano or guitar, no vocals please")

Parsed preferences: {"genres": ["Instrumental"], "exclude_genres": [], "include_tags": ["Piano", "Acoustic guitar", "Ambient music", "Background music"], "exclude_tags": ["Singing", "Choir", "Male singing", "Female singing", "Child singing", "Synthetic singing", "Rapping", "Humming"]}

These instrumental picks offer a calm study-friendly mix of piano, acoustic guitar, and gentle ambient background sounds, with no vocal tags.

#526 | Instrumental | Instrumental piano and ambient tags make this a fitting piano-centered choice.


<audio player removed to keep the notebook small; re-run the cell to listen>

#670 | Instrumental | Acoustic guitar and plucked-string tags bring in the requested guitar sound.


<audio player removed to keep the notebook small; re-run the cell to listen>

#274 | Instrumental | Piano, lullaby, and tender-music tags suggest a gentle, soothing option.


<audio player removed to keep the notebook small; re-run the cell to listen>

#403 | Instrumental | Tender ambient and background-music tags suit quiet studying.


<audio player removed to keep the notebook small; re-run the cell to listen>

#316 | Instrumental | Ambient, tender, and lullaby tags point to a soft instrumental backdrop.


<audio player removed to keep the notebook small; re-run the cell to listen>

,song_id,genre,tags,score,reason
526,6199,Instrumental,"[Keyboard (musical), Piano, Classical music, A...",4.59,Instrumental piano and ambient tags make this ...
670,7042,Instrumental,"[Acoustic guitar, Guitar, Plucked string instr...",4.31,Acoustic guitar and plucked-string tags bring ...
274,14613,Instrumental,"[Electric piano, Keyboard (musical), Piano, Lu...",4.23,"Piano, lullaby, and tender-music tags suggest ..."
403,22583,Instrumental,"[New-age music, Ambient music, Tender music, B...",4.63,Tender ambient and background-music tags suit ...
316,6718,Instrumental,"[New-age music, Ambient music, Tender music, S...",4.21,"Ambient, tender, and lullaby tags point to a s..."


In [45]:
recommend("High energy music for the gym: heavy drums, electric guitar, aggressive vibe")

Parsed preferences: {"genres": ["Rock"], "exclude_genres": [], "include_tags": ["Drum kit", "Electric guitar", "Heavy metal", "Exciting music", "Angry music"], "exclude_tags": []}

A high-energy rock selection leans into heavy metal, exciting and angry vibes, with electric-guitar tracks for a forceful gym soundtrack.

#156 | Rock | Its heavy metal, punk rock, and exciting-music tags make it a strong high-energy gym fit.


<audio player removed to keep the notebook small; re-run the cell to listen>

#761 | Rock | Heavy metal and electric guitar combine with exciting music for an aggressive rock workout feel.


<audio player removed to keep the notebook small; re-run the cell to listen>

#831 | Rock | Punk rock, heavy metal, and angry-music tags match the requested intensity.


<audio player removed to keep the notebook small; re-run the cell to listen>

#974 | Rock | Heavy metal and angry music give this rock pick a forceful, energizing edge.


<audio player removed to keep the notebook small; re-run the cell to listen>

#921 | Rock | Its electric guitar and heavy metal tags deliver the guitar-driven workout sound you want.


<audio player removed to keep the notebook small; re-run the cell to listen>

,song_id,genre,tags,score,reason
156,9150,Rock,"[Heavy metal, Rock and roll, Progressive rock,...",4.74,"Its heavy metal, punk rock, and exciting-music..."
761,18812,Rock,"[Electric guitar, Progressive rock, Heavy meta...",4.67,Heavy metal and electric guitar combine with e...
831,3381,Rock,"[Punk rock, Progressive rock, Heavy metal, Ang...",4.25,"Punk rock, heavy metal, and angry-music tags m..."
974,17965,Rock,"[Heavy metal, Angry music, Song, Punk rock, Da...",4.20,Heavy metal and angry music give this rock pic...
921,20420,Rock,"[Electric guitar, Heavy metal, Guitar, Rock an...",4.08,Its electric guitar and heavy metal tags deliv...


**2. "I liked these songs"** — audio similarity to liked tracks, optionally steered by text.

In [46]:
# pretend the listener liked two tracks from the first genre
liked = catalog.index[catalog.genre == GENRES[0]][:2].tolist()
print("Liked:"); play(liked)
print()
recommend("more like these, but a bit more upbeat", liked_ids=liked)

Liked:
#5 | Chiptune / Glitch | Electronic music, Dubstep, Electronica, Synthesizer


<audio player removed to keep the notebook small; re-run the cell to listen>

#10 | Chiptune / Glitch | Drum and bass, Angry music, Electronic music, Dubstep


<audio player removed to keep the notebook small; re-run the cell to listen>


Parsed preferences: {"genres": [], "exclude_genres": [], "include_tags": ["Happy music", "Exciting music", "Dance music"], "exclude_tags": []}

These picks keep the electronic, chiptune-adjacent feel while leaning into dance grooves and more exciting energy.

#880 | Chiptune / Glitch | Chiptune / Glitch with disco-house, dance, and exciting tags makes this a close, upbeat match.


<audio player removed to keep the notebook small; re-run the cell to listen>

... (5 outputs left out)

<audio player removed to keep the notebook small; re-run the cell to listen>

#521 | Electronic | Drum and bass connects to one of your liked tracks, while dance, house, and Afrobeat add rhythmic variety.


<audio player removed to keep the notebook small; re-run the cell to listen>

,song_id,genre,tags,score,reason
880,24160,Chiptune / Glitch,"[Disco, House music, Dance music, Electronica,...",3.87,"Chiptune / Glitch with disco-house, dance, and..."
691,3629,Chiptune / Glitch,"[Disco, Techno, Dance music, Soundtrack music,...",3.74,This Chiptune / Glitch pick pairs techno and d...
584,17379,International,"[Disco, Pop music, Dance music, Exciting music...",3.50,A change of genre to International brings disc...
736,17523,Chiptune / Glitch,"[Disco, Electronic music, Electronic dance mus...",3.31,Chiptune / Glitch stays close to your examples...
521,21174,Electronic,"[Electronic dance music, Drum and bass, Dance ...",3.04,Drum and bass connects to one of your liked tr...


**3. Your own audio file** — drop any mp3/wav in `data/` (falls back to a catalogue clip so the cell always runs).

In [47]:
MY_SONG = "data/my_song.mp3"
liked_audio = MY_SONG if os.path.exists(MY_SONG) else music[5]["audio"]["array"]  # demo clip: it will find itself first
recommend(liked_audio=liked_audio, n=3)

Parsed preferences: {"genres": [], "exclude_genres": [], "include_tags": [], "exclude_tags": []}

With no stated preferences to narrow the field, these picks offer a varied starting point across electronic styles.

#5 | Chiptune / Glitch | Its Chiptune / Glitch genre and tags spanning dubstep, techno, and synthesizer suggest a lively electronic mix.


<audio player removed to keep the notebook small; re-run the cell to listen>

#764 | Experimental | This Experimental pick leans on synthesizer, sampler, and keyboard sounds for a more exploratory electronic direction.


<audio player removed to keep the notebook small; re-run the cell to listen>

#890 | Electronic | Its Electronic genre and ambient-music tags add a calmer contrast to the more beat-driven selections.


<audio player removed to keep the notebook small; re-run the cell to listen>

,song_id,genre,tags,score,reason
5,11704,Chiptune / Glitch,"[Electronic music, Dubstep, Electronica, Synth...",3.00,Its Chiptune / Glitch genre and tags spanning ...
764,9333,Experimental,"[Synthesizer, Sampler, Electronic music, Elect...",2.45,"This Experimental pick leans on synthesizer, s..."
890,204,Electronic,"[Synthesizer, Ambient music, Electronica, Elec...",2.36,Its Electronic genre and ambient-music tags ad...


**4. Batch / evaluation:** do genre requests return that genre? (A quick check of the whole pipeline, run without audio.)

In [48]:
rows = []
for g in GENRES:
    recs = recommend(f"I want {g} music", n=5, play_audio=False, show_prefs=False)
    rows.append({"requested": g, "hit_rate": (recs.genre == g).mean()})
pd.DataFrame(rows).set_index("requested")


A varied selection of chiptune and glitch tracks with electronic, video-game, and soundtrack character.


A varied selection of electronic tracks spanning dubstep, drum and bass, trance, techno, house, and disco.


These Experimental tracks offer a varied mix of unusual bells, electronic textures, unconventional wind sounds, resonant percussion, and plucked strings.


These five Folk tracks offer a mix of global folk influences, plucked-string textures, and fiddle- and bluegrass-oriented sounds.


A varied set of Hip-Hop picks spanning reggae and Afrobeat influences, beatboxing, and dance-oriented sounds.


These instrumental picks offer a varied mix of ambient textures, soundtrack moods, and acoustic and orchestral sounds.


This selection explores International music across Afrobeat and dance, Latin rhythms, traditional folk, flamenco, and electronic sounds.


These picks center on Pop music while offering a mix of upbeat, soulful, R&B, and dance-influenced sounds.


These picks cen

,hit_rate
requested,
Chiptune / Glitch,1.0
Electronic,1.0
Experimental,1.0
Folk,1.0
Hip-Hop,1.0
Instrumental,1.0
International,1.0
Pop,1.0
Punk,1.0


## Wrap‑up & ideas to extend
- **Why it works:** PANNs embeddings capture *how a track sounds*; its AudioSet tags give a vocabulary the LLM can map natural language onto — so text queries need no extra text‑audio model. GPT Luna is used only where language understanding helps (query parsing and explaining picks), and it's grounded: it can only return ids that local retrieval found.
- **Swap the model:** set `OPENAI_MODEL` (e.g. `gpt-5.6-luna`, `gpt-6-sol`) — the Responses API call is identical.
- **More data:** any 🤗 audio dataset with a label column drops in (e.g. GTZAN from the [ai‑audio‑datasets list](https://github.com/Yuan-ManX/ai-audio-datasets)); concatenate catalogues and re-run the embedding cell.
- **Scale:** replace `LocalVectorIndex` with the Pinecone index from Part 1 (upsert `music_emb` with `genre`/`tags` as metadata) or FAISS.
- **Better text→audio:** add a CLAP model (joint text/audio embeddings) as another retrieval signal.